# Language Modelling 1

My notes for the following videos by Andrej Karpathy:

- [_The spelled-out intro to language modeling: building makemore_](https://www.youtube.com/watch?v=PaCmpygFfXo)
- [_Building makemore Part 2: MLP_](https://www.youtube.com/watch?v=TCH_1BHY58I)

We will build models of increasing complexity that attempt to generate text similar to some given input text.

## Bigram Model: Counts

In a bigram model, words are generated character by character. At any point, the next character of the sequence is generated based on the likelihood of characters following the current character.

For each word, we introduce the '.' as a special start/end character. Then, the frequency table is generated by iterating over all words and counting the occurrence of each pair of characters. This is stored in a 27-by-27 matrix $P$ which is then normalized. Hence, each row of this matrix is a probability distribution $p_c$ corresponding to the character that follows the row character $c$.

The loop to generate text is quite intuitive. We start at a '.' and generate random letters according to their probability distributions until another '.' is reached, at which we end the loop. The result is better than if we sampled randomly, but is still poor overall.

In [46]:
import torch

words = open('names.txt', 'r').read().splitlines()

# Get all possible characters and create char to int mapping
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

N = torch.zeros((27, 27), dtype=torch.int32)
for w in words:
    # Add start and end character
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1

# Normalize rows
P = N.float()
P /= P.sum(1, keepdim=True)


g = torch.Generator().manual_seed(77777777)

for i in range(20):
    out = []
    ix = 0
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    print(''.join(out), end=" ")

brn. asto. am. dalydaisava. aniaman. onidavynianla. ah. jaiatisihanonyel. pr. kakynielya. tickacou. zallatt. k. iliaa. brilelih. annn. kiyr. a. lmyngameire. mbrelia. 

## Bigram Model: Neural Network

We now aim to train a simple neural network to obtain the weights of the $P$ matrix, as opposed to using count data. This first requires defining an appropriate loss function now that the data is non-numeric.

### Loss Function

Given some sequence of of characters $(c_1, ..., c_{n+1})$ that belongs to the input data, we would expect a good model to assign high probabilities to each pair $(c_i, c_{i+1})$, meaning $p_{c_i} (c_{i+1})$ should be substantial. Similar to MLE theory in classical statistics, we can summarize the performance of a model with parameters $\mathbf{p}$ using a log-likelihood function. This is chosen for convenience as a product of many probabilities would result in a very small number. Instead, applying the log to this product of probabilities reduces the product to a sum of reasonably large numbers. Aiming for an interpretable minimization problem, this log-likelihood is negated and normalized. 

$$
    L(\mathbf{p}) = \frac{1}{n} \sum_{i=1}^{n} - \log(p_{c_i} (c_{i+1}))
$$

A model assigning equal probability to every character would have an $L$-value of $\log 27 \approx 3.30$, which can be interpreted as random guessing. Below this value would indicate that the model predicts meaningfully, and a value of 0 would indicate a "perfect" model (however this may also indicate overfitting). Because of a risk of taking a log of 0 is potentially undesirable, artificial counts may be added to every entry in the matrix $P$ before normalizing, called _model smoothing_.

### Encoding

The way to model this problem using a neural network may not be clear at first sight. Because weights act multiplicatively on values, having inputs take on the integer values corresponding to characters is not a good choice. For this reason, a _one-hot_ encoding is used. Concretely, the one-hot encoding for a character $i$ is the 27-dimensional vector with value $1$ at component $i$ and zeros elsewhere. Given a character encoding $\mathbf{x}$, the neural network will output a 27-dimensional vector $\mathbf{y}$.

The neural network does not have the mechanism to ensure that $\mathbf{y}$ is a valid probability distribution. The easiest workaround is to interprect the values of $\mathbf{y}$ as _log counts_. Similar to how we counted the occurances of bigrams to construct the pre-normalized matrix $P$, exponentiating these _log counts_ simulates a pre-normalized matrix $P$. Given a one-hot encoding $\mathbf{x}_c$, let $\mathbf{y} = \exp[\mathbf{x}_c^T W]$, where $\exp(\cdot)$ is applied component-wise. The product $\mathbf{x}_c^T W$ results in exactly row $c$ in $W$, which when exponentiated gives an equivalent of row $c$ in the pre-normalized matrix $P$ from above. Normalizing $\mathbf{y}$ results in a valid probability distribution. We can parallelize this process for many $\mathbf{x}_c$'s using matrix multiplication.

Converting from an output layer of real numbers $\mathbf{z}$ to a probability distribution is known as the _softmax_ function,

$$
    \sigma(\mathbf{z})_i = \frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}}.
$$
Feeding this into the loss function defined above yields a chain of differentiable functions that depend on $W$, to which we can apply backpropagation and use gradient descent to optimize the entries of $W$.

### Regularization

To encourage the model to generalize instead of memorizing the training data, it is often desirable to add a _regularization_ term to the loss function. In this case, we can add a factor proportional to the sum of squared entries of $W$. This "pushes" $W$ towards having all $0$ entries (i.e. a uniform distribution for all rows of $P$). 

### Performance

Since this NN does not use any more information than the previous simpler model, the performance of the NN does not beat the performance of the simpler model. In fact, the simpler model is similar to an MLE for the given input data, and hence performs quite well compared to other choices of matrices. For better performance, it would be better to consider more _context_, i.e. previous characters.

In [87]:
import torch.nn.functional as F

# Create training set of bigrams (x, y)
xs, ys = [], []
for w in words:
    # Add start and end character
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()

xenc = F.one_hot(xs, num_classes=27).float()
yenc = F.one_hot(ys, num_classes=27).float()

# Initialize NN
W = torch.randn((27, 27), requires_grad=True)

# Gradient Descent
for i in range(100):
    # Forward pass
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdims=True)
    
    # Loss with regularization
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W**2).mean()
    
    # Backwards pass
    W.grad = None
    loss.backward()
    W.data += -50 * W.grad

# Sample
for i in range(10):
    out = []
    ix = 0
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W
        counts = logits.exp()
        p = counts / counts.sum(1, keepdims=True)
        ix = torch.multinomial(p, num_samples=1, replacement=True).item()
        out.append(itos[ix])
        if ix == 0:
            break
            
    print(''.join(out), end=' ')

hiaeadelyasalloulaque. kx. jaxge. soresxona. beh. 

## n-Gram: Neural Network

The next logical step would be to make a similar model using 2 or more letters of context. Unfortunately, even with a large amount of training data, the counts per $n$-gram quickly decrease and thus less and less meaningful data can be extracted. Instead, we choose a number of features $m$ and encode every letter as a vector in $\mathbb{R}^m$. Ideally, letters with similar "meanings" would be close to each other in this $m$-dimensional space. The neural network then take in $n-grams$ and determine what letter should come after it. Similarly to how we prepare the bigrams for training in the previous examples, we can prepare the $n$ letter context training data. As a note, this idea was originally proposed for words, but is simpler to demonstrate using letters.

The idea behind this model comes from [Bengio et al., 2003](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf).

### Setup

We first build a $27 \times m$ embedding matrix $C$ such that row vector $i$ will be the encoding of character $i$. The first layer of the NN contains $k$ neurons, each with $nm$ inputs corresponding to the $n$ input letters, each with embeddings of $m$ components. The output layer will contain 27 neurons, each with $nm$ inputs. This last layer will then be interpreted as logits, fed into a softmax to produce a probability distribution on the 27 characters. This NN can be easily represented as a chain of matrix multiplications and component-wise activation functions, and using PyTorch tensors gives a way to parallelize the entire process.

### Splitting Data

As the NN parameter count increases, the NN becomes more and more capable of overfitting the data, especially if the data set is relatively small. In other words, the loss on the data set will likely be very small, yet the model will perform poorly on examples that were not originally in the data set. For this reason, it is common to split the data:

1. The _training split_ contains roughly 80% of the data and is used to optimize the parameters of the NN (i.e. weights and biases)
2. The _validation_ or _dev_ split contains roughly 10% of the data and is used for development with regards to the hyperparameters (size of hidden layers, embeddings, regularization strength, etc.)
3. The _test_ split also contains roughly 10% of the data and is used to evaluate the performance of the model. This is data that the NN has not seen yet, and hence should ideally combat overfitting.

### Training and Optimization

At this point, we begin to see performance issues in evaluating the model on the whole data set every time. As previously mentioned, selecting a random subset of the data (a batch) on which to calculate the gradient will aleviate this issue. 

It is also not obvious how to choose the learning rate $\alpha$. Here, we can iterate through some logarithmically spaced learning rates and perform an iteration with each to attempt to find a learning rate. Plotting this, as seen below, allows to gaugue where the learning rate can be large enough without causing divergence. In this case $\alpha = 0.1$ comes right before a plateau and seems to be a good choice. Unfortunatelly, this "approximate gauging" is one of the best ways to do so and because of the complexity of the model, it is impossible to determine anything precise. However, this is good enough in practice.

<img 
    style="display: block; 
           margin-left: auto;
           margin-right: auto;
           width: 30%;"
    src="lrrangetest.png" 
    alt="alpha">
</img>

### Diagnosing Performance Issues

The initial choice of hyperparameters was embedding dimension $m = 2$ and hidden layer size $k = 100$.

After training, the model still underfits the data, since the performance on the train and test splits are roughly equal. After assuring that number of descent iterations and learning rate are chosen properly, this is an issue related to the size of the model. In fact, the model is not even powerful enough to "memorize" the training data. Hence, we update the hidden layer size $k$ from 100 to 300 neurons.

Despite this update (and finding an appropriate step size for SGD), the model still does not seem to perform well after training. It is also possible that our original choice of character embedding dimension $m = 2$ was too low. Luckily, this low dimension allows for an interpretable plot. There is some noticeable structure: vowels are clustered together, and odd characters such as '.' and 'q' are distinctly similar. However, many other letters are still clustered together and the model likely has difficulty differentiating between them. 


<img 
    style="display: block; 
           margin-left: auto;
           margin-right: auto;
           width: 45%;"
    src="letterembedding.png" 
    alt="alpha">
</img>

We therefore increase embedding dimension to $m = 10$ (and decrease hidden layer size to $k = 200$ to keep the model of reasonable size). After training on this updated network, the loss is indeed lower than it was in the case of $m = 2$, although seems as if the peformance on the training split is slightly better than on the dev split, which indicates a slight overfit. For reference, Andrej obtains a loss of $2.17$ with $m = 10$ and $k = 200$ and invites us to experiment with hyperparameters to try and obtain a lower loss. At this point, we have a couple options: 

1. Continue tuning optimization
2. Continue changing hyperparameters $m, k$
3. Increase context length $n$

The adjustment of these three factors is at the heart of developing a successful model. Again, the complexity of the model means that often a "trial and error" approach is the only way to go about improving it.

I have adjusted

- $n = 3$, since longer context lengths are not necessarily effective in this scenario since the names we are working with are often short.
- $m = 11$, $k = 300$, obtained through trial and error.
- SGD batch size 50, to hopefully obtain a higher quality gradient
- 250000 iterations, the first 100000 of which with $\alpha = 0.1$, the rest with $\alpha = 0.01$

This results in loss $L \approx 2.03$ on the training data and $L \approx 2.13$ on both the dev and test data. While this indicates an overfit to the data, the performance on unseen data is still sufficiently good. When sampled from, the model produces names that are beginning to look like real names.

In [310]:
# BUILD AND SPLIT DATA

# Hyperparameters
n = 3 # context length
m = 11 # embedding dimension
k = 300 # hidden layer size

def build_dataset(words):
    block_size = n # context length
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X  = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [311]:
# BUILD NN

C = torch.randn((27, m))
W1 = torch.randn((m * n, k)) # second parameter is hidden layer size
b1 = torch.randn(k)
W2 = torch.randn(k, 27)
b2 = torch.randn(27)
parameters = [C, W1, b1, W2, b2]

for p in parameters:
    p.requires_grad = True

In [321]:
# TRAINING

sgd_batch_size = 50
max_steps = 250000

# Keep track of learning rate performance and/or optimization
# lre = torch.linspace(-3, 0, 1000)
# lrs = 10**lre

lri = []
lossi = []
stepi = []

for i in range(max_steps):
    # SGD batch selection
    ix = torch.randint(0, Xtr.shape[0], (sgd_batch_size,))
    
    # Forward Pass
    emb = C[Xtr[ix]] # Input (32, 3, 2)
    h = torch.tanh(emb.view(-1, m * n) @ W1 + b1) # First layer output (32, 100)
    logits = h @ W2 + b2 # Second layer output (32, 27)
    loss = F.cross_entropy(logits, Ytr[ix]) # this is the standard loss we have been using
    
    # Backward Pass
    for p in parameters:
        p.grad = None
    loss.backward()
    
    # Update using custom lr
    # lr = lrs[i]
    lr = 0.1 if i < 100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    # Track lr stats
    # lri.append(lre[i])
    stepi.append(i)
    lossi.append(loss.log10().item())

    if i % 25000 == 0 and i != 0:
        print(f"iter: {i}, averaged log loss: {sum(lossi[i-1000:i])/1000:.8f}")

iter: 10000, averaged loss: 0.4208
iter: 20000, averaged loss: 0.3927
iter: 30000, averaged loss: 0.3812
iter: 40000, averaged loss: 0.3730
iter: 50000, averaged loss: 0.3678
iter: 60000, averaged loss: 0.3615
iter: 70000, averaged loss: 0.3577
iter: 80000, averaged loss: 0.3541
iter: 90000, averaged loss: 0.3530
iter: 100000, averaged loss: 0.3500
iter: 110000, averaged loss: 0.3208
iter: 120000, averaged loss: 0.3211
iter: 130000, averaged loss: 0.3203
iter: 140000, averaged loss: 0.3186
iter: 150000, averaged loss: 0.3208
iter: 160000, averaged loss: 0.3203
iter: 170000, averaged loss: 0.3169
iter: 180000, averaged loss: 0.3188
iter: 190000, averaged loss: 0.3184
iter: 200000, averaged loss: 0.3163
iter: 210000, averaged loss: 0.3474
iter: 220000, averaged loss: 0.3463
iter: 230000, averaged loss: 0.3427
iter: 240000, averaged loss: 0.3421


In [332]:
# EVALUATION

@torch.no_grad()
def split_loss(split):
    x, y = {
        'train': (Xtr, Ytr),
        'val': (Xdev, Ydev),
        'test': (Xte, Yte),
    }[split]
    emb = C[x]
    embcat = emb.view(emb.shape[0], -1)
    h = torch.tanh(embcat @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)
    print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.0293610095977783
val 2.1302735805511475


In [331]:
# SAMPLING FROM THE MODEL

for _ in range(20):
    out = []
    context = [0] * n
    while True:
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break
    print(''.join(itos[i] for i in out), end=' ')

alienydono. giulia. and. auriston. maker. jermiant. jareda. alom. jatha. jaiya. jaystonge. zyanimit. mad. hayson. haila. beshana. eza. imo. josyn. izy. 